In [1]:
!pip install transformers
!pip install sentencepiece

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.3 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 35.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 41.5 MB/s eta 0:00:00


In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

import transformers
from transformers import AutoTokenizer, RobertaTokenizerFast, RobertaForTokenClassification
from torch.optim import AdamW

import torch
import torch.nn as nn
from torch.optim import SGD
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score,f1_score, precision_score, recall_score

In [6]:
class DistilbertNER(nn.Module):
    """
    Implement NN class based on distilroberta pretrained from Hugging face.
    Inputs : 
      tokens_dim : int specifying the dimension of the classifier
    """
    
    def __init__(self, tokens_dim):
        super(DistilbertNER, self).__init__()
        
        if type(tokens_dim) != int:
            raise TypeError('Please tokens_dim should be an integer')

        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        self.pretrained = RobertaForTokenClassification.from_pretrained(
            "distilroberta-base", num_labels=tokens_dim)  # Set the output of each token classifier = unique_labels

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Forward computation of the network
        Input:
          - input_ids : from model tokenizer
          - attention_mask : mask from model tokenizer
          - labels : if given the model is able to return the loss value
        """

        if labels is None:  # Inference time: no labels
            return self.pretrained(input_ids=input_ids, attention_mask=attention_mask)

        return self.pretrained(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

In [29]:
df_new = pd.read_csv('./NEW_Annotated_Dataset_NER.csv')


In [30]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df_new, test_size=0.2, random_state=42)

df_dev = df_test  # Use test set as dev set for simplicity

# %% Tokenizer Setup
tokenizer = RobertaTokenizerFast.from_pretrained("distilroberta-base")

In [31]:
import re

# Function to clean the labels
def clean_labels(label_column):
    cleaned_labels = []
    for label_list in label_column:
        # Remove square brackets, single quotes, and extra spaces
        cleaned_label = re.sub(r"[\[\]']", "", label_list)
        cleaned_label = cleaned_label.replace(",", "")  # Remove commas if present
        cleaned_labels.append(cleaned_label.strip())  # Remove leading/trailing spaces
    return cleaned_labels

# Apply the cleaning function to the 'Labels' column
df_train["Labels"] = clean_labels(df_train["Labels"])
df_test["Labels"] = clean_labels(df_test["Labels"])

# Check the unique cleaned labels
print(df_train["Labels"].unique())

['QUANTITY NAME O STATE STATE O'
 'QUANTITY UNIT DF NAME FORM O STATE STATE'
 'O O O O O O O O O O O O O O O O' ... 'QUANTITY UNIT O O NAME O O O'
 'QUANTITY UNIT O NAME NAME O O O O O O O O O O NAME O O O O O'
 'QUANTITY NAME NAME O QUANTITY O O O']


In [21]:
class NerDataset(torch.utils.data.Dataset):
    """
    Custom dataset implementation to get (text, labels) tuples
    """
    def __init__(self, df):
        if not isinstance(df, pd.DataFrame):
            raise TypeError('Input should be a dataframe')

        # Map new dataset's columns to texts and labels
        texts = df["Ingredient_Phrases"].values.tolist()
        tags_list = [i.split() for i in df["Labels"].values.tolist()]

        self.texts = [tokenizer(text, padding="max_length", truncation=True, return_tensors="pt") for text in texts]
        self.labels = [match_tokens_labels(text, tags) for text, tags in zip(self.texts, tags_list)]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        batch_text = self.texts[idx]
        batch_labels = self.labels[idx]

        return batch_text, torch.LongTensor(batch_labels)

In [22]:
class MetricsTracking:
    """
    Tracks accuracy, F1-score, precision, and recall for NER tasks
    """
    def __init__(self):
        self.total_acc = 0
        self.total_f1 = 0
        self.total_precision = 0
        self.total_recall = 0

    def update(self, predictions, labels, ignore_token=-100):
        predictions = predictions.flatten()
        labels = labels.flatten()

        predictions = predictions[labels != ignore_token]
        labels = labels[labels != ignore_token]

        predictions = predictions.to("cpu")
        labels = labels.to("cpu")

        acc = accuracy_score(labels, predictions)
        f1 = f1_score(labels, predictions, average="macro")
        precision = precision_score(labels, predictions, average="macro")
        recall = recall_score(labels, predictions, average="macro")

        self.total_acc += acc
        self.total_f1 += f1
        self.total_precision += precision
        self.total_recall += recall

    def return_avg_metrics(self, data_loader_size):
        n = data_loader_size
        metrics = {
            "acc": round(self.total_acc / n, 3),
            "f1": round(self.total_f1 / n, 3),
            "precision": round(self.total_precision / n, 3),
            "recall": round(self.total_recall / n, 3)
        }
        return metrics

In [14]:
def match_tokens_labels(tokenized_input, tags, ignore_token=-100):
    word_ids = tokenized_input.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(ignore_token)
        else:
            try:
                reference_tag = tags[word_idx]
                label_ids.append(tag2idx[reference_tag])
            except:
                label_ids.append(ignore_token)

        previous_word_idx = word_idx

    return label_ids
    
def tags_mapping(tags_series):
    unique_tags = set()

    for tag_list in tags_series:
        for tag in tag_list.split():
            unique_tags.add(tag)

    tag2idx = {k: v for v, k in enumerate(sorted(unique_tags))}
    idx2tag = {v: k for v, k in tag2idx.items()}

    unseen_label = tag2idx["O"]

    return tag2idx, idx2tag, unseen_label, unique_tags


In [17]:
def train_loop(model, train_dataset, dev_dataset, optimizer, batch_size, epochs):
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    dev_dataloader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_metrics_list = {'loss': [], 'acc': [], 'precision': [], 'recall': [], 'f1': []}
    test_metrics_list = {'loss': [], 'acc': [], 'precision': [], 'recall': [], 'f1': []}

    for epoch in range(epochs):
        train_metrics = MetricsTracking()
        total_loss_train = 0

        model.train()  # Training mode

        for train_data, train_label in tqdm(train_dataloader):
            train_label = train_label.to(device)
            mask = train_data['attention_mask'].squeeze(1).to(device)
            input_id = train_data['input_ids'].squeeze(1).to(device)

            optimizer.zero_grad()

            output = model(input_id, mask, train_label)
            loss, logits = output.loss, output.logits
            predictions = logits.argmax(dim=-1)

            train_metrics.update(predictions, train_label)
            total_loss_train += loss.item()

            # Gradient step
            loss.backward()
            optimizer.step()

        # Evaluation Mode
        model.eval()
        dev_metrics = MetricsTracking()
        total_loss_dev = 0

        with torch.no_grad():
            for dev_data, dev_label in dev_dataloader:
                dev_label = dev_label.to(device)

                mask = dev_data['attention_mask'].squeeze(1).to(device)
                input_id = dev_data['input_ids'].squeeze(1).to(device)

                output = model(input_id, mask, dev_label)
                loss, logits = output.loss, output.logits

                predictions = logits.argmax(dim=-1)

                dev_metrics.update(predictions, dev_label)
                total_loss_dev += loss.item()

        train_results = train_metrics.return_avg_metrics(len(train_dataloader))
        dev_results = dev_metrics.return_avg_metrics(len(dev_dataloader))

        train_metrics_list['loss'].append(total_loss_train / len(train_dataset))
        train_metrics_list['acc'].append(train_results['acc'])
        train_metrics_list['precision'].append(train_results['precision'])
        train_metrics_list['recall'].append(train_results['recall'])
        train_metrics_list['f1'].append(train_results['f1'])

        test_metrics_list['loss'].append(total_loss_dev / len(dev_dataset))
        test_metrics_list['acc'].append(dev_results['acc'])
        test_metrics_list['precision'].append(dev_results['precision'])
        test_metrics_list['recall'].append(dev_results['recall'])
        test_metrics_list['f1'].append(dev_results['f1'])

        print(f"TRAIN\nLoss: {total_loss_train / len(train_dataset)}\nMetrics: {train_results}\n")
        print(f"VALIDATION\nLoss: {total_loss_dev / len(dev_dataset)}\nMetrics: {dev_results}\n")

    return train_metrics_list, test_metrics_list


In [32]:
# %% Create tag-label mapping
tag2idx, idx2tag, unseen_label, unique_tags = tags_mapping(df_train["Labels"])

In [36]:
# Helper function for converting tags to label indices
def tags_2_labels(tags: str, tag2idx: dict):
    return [tag2idx[tag] if tag in tag2idx else unseen_label for tag in tags.split()]
for df in [df_train, df_dev, df_test]:
    df["labels"] = df["Labels"].apply(lambda tags: tags_2_labels(tags, tag2idx))

In [38]:
# Step 1: Ensure all entries in 'Ingredient_Phrases' are strings
df_train["Ingredient_Phrases"] = df_train["Ingredient_Phrases"].astype(str)

# Step 2: Remove any empty or None values
df_train = df_train[df_train["Ingredient_Phrases"].str.strip() != '']  # Remove empty strings
df_train = df_train.dropna(subset=["Ingredient_Phrases"])  # Remove None or NaN values

# Now try tokenizing again
text_tokenized = tokenizer(df_train["Ingredient_Phrases"].values.tolist(), 
                           padding='max_length', truncation=True, return_tensors="pt")


In [39]:
# %% Tokenize the dataset
tokenizer = RobertaTokenizerFast.from_pretrained("distilroberta-base")
text_tokenized = tokenizer(df_train["Ingredient_Phrases"].values.tolist(), padding='max_length', truncation=True, return_tensors="pt")

# %% Create the model
model = DistilbertNER(len(unique_tags))  # Number of unique tags will define the classifier dimension

# Create datasets
train_dataset = NerDataset(df_train)
dev_dataset = NerDataset(df_dev)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
# Define optimizer
lr = 1e-2
optimizer = SGD(model.parameters(), lr=lr, momentum=0.9)

# %% Start the training loop
parameters = {
    "model": model,
    "train_dataset": train_dataset,
    "dev_dataset": dev_dataset,
    "optimizer": optimizer,
    "batch_size": 92,
    "epochs": 8
}

train_metrics_list, test_metrics_list = train_loop(**parameters)



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 83/83 [03:01<00:00,  2.19s/it]


TRAIN
Loss: 0.008690999173804333
Metrics: {'acc': 0.729, 'f1': 0.525, 'precision': 0.582, 'recall': 0.521}

VALIDATION
Loss: 0.005077521314746455
Metrics: {'acc': 0.849, 'f1': 0.725, 'precision': 0.765, 'recall': 0.72}



100%|██████████| 83/83 [03:00<00:00,  2.18s/it]


TRAIN
Loss: 0.004630895382479618
Metrics: {'acc': 0.859, 'f1': 0.758, 'precision': 0.825, 'recall': 0.738}

VALIDATION
Loss: 0.004010596636094545
Metrics: {'acc': 0.885, 'f1': 0.815, 'precision': 0.87, 'recall': 0.803}



100%|██████████| 83/83 [03:01<00:00,  2.18s/it]


...

TRAIN
Loss: 0.00286318591158641
Metrics: {'acc': 0.911, 'f1': 0.858, 'precision': 0.883, 'recall': 0.855}

VALIDATION
Loss: 0.0031825330696607893
Metrics: {'acc': 0.911, 'f1': 0.89, 'precision': 0.884, 'recall': 0.91}



100%|██████████| 83/83 [03:12<00:00,  2.32s/it]


TRAIN
Loss: 0.0026269424922372164
Metrics: {'acc': 0.917, 'f1': 0.872, 'precision': 0.888, 'recall': 0.872}

VALIDATION
Loss: 0.0032751788355802236
Metrics: {'acc': 0.907, 'f1': 0.88, 'precision': 0.879, 'recall': 0.895}



100%|██████████| 83/83 [03:09<00:00,  2.29s/it]


TRAIN
Loss: 0.0025119121823655934
Metrics: {'acc': 0.92, 'f1': 0.874, 'precision': 0.885, 'recall': 0.875}

VALIDATION
Loss: 0.003102199286222458
Metrics: {'acc': 0.914, 'f1': 0.889, 'precision': 0.881, 'recall': 0.909}



# New Code

In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, RobertaForTokenClassification, AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from transformers import RobertaForTokenClassification
import torch.nn as nn

class DistilRoBERTaNER(nn.Module):
    def __init__(self, tokens_dim):
        super(DistilRoBERTaNER, self).__init__()
        
        if type(tokens_dim) != int:
            raise TypeError('tokens_dim should be an integer')
        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        # Load the pre-trained Roberta model for token classification with specified number of labels
        self.pretrained = RobertaForTokenClassification.from_pretrained(
            "distilroberta-base", num_labels=tokens_dim
        )

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Forward pass of the network.
        """
        if labels is None:
            return self.pretrained(input_ids=input_ids, attention_mask=attention_mask)
        return self.pretrained(input_ids=input_ids, attention_mask=attention_mask, labels=labels)


# Update the tokenizer initialization in the main function
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

/home/ritisha21089/.local/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
# %% Custom dataset class for NER using DistilBERT
class NerDataset(Dataset):
    """
    Custom dataset class for NER tasks.
    """

    def __init__(self, dataframe, tokenizer, label_map, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        # Ensure the 'Ingredient_Phrases' column is a string
        text = str(row['Ingredient_Phrases'])

        labels = row['Labels']

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            return_attention_mask=True,
            is_split_into_words=False
        )

        # Convert labels to match tokenization
        label_ids = [-100] * self.max_len  # Default padding label
        word_ids = encoding.word_ids()

        labels_list = eval(labels)  # Convert label string back to list

        # Assign NER labels for each word
        for i, word_id in enumerate(word_ids):
            if word_id is None or word_id >= len(labels_list):
                label_ids[i] = -100
            else:
                label_ids[i] = self.label_map.get(labels_list[word_id], -100)

        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        label_ids = torch.tensor(label_ids)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_ids
        }

# %% Label mapping for the NER task
label_map = {
    'QUANTITY': 0,
    'UNIT': 1,
    'NAME': 2,
    'FORM': 3,
    'STATE': 4,
    'DRY/FRESH': 5,
    'SIZE': 6,
    'PREPROCESSING': 7,
    'O': -100  # Outside token or padding
}

        

In [9]:
# %% Load the new dataset and tokenizer
df = pd.read_csv('/home/ritisha21089/NEW_Annotated_Dataset_NER.csv')

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
#data_loader = DataLoader(ner_dataset, batch_size=16, shuffle=True)

train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


# %% Instantiate the model
model = DistilRoBERTaNER(tokens_dim=len(label_map))

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# %% Training loop (basic example)
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training settings
epochs = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

/home/ritisha21089/.local/lib/python3.8/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


DistilRoBERTaNER(
  (pretrained): RobertaForTokenClassification(
    (roberta): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(50265, 768, padding_idx=1)
        (position_embeddings): Embedding(514, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-5): 6 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (dense

In [11]:

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        # Extract the input ids, attention masks, and labels from the batch
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# %% Evaluation (example metrics calculation)
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        logits = logits.detach().cpu().numpy()
        label_ids = labels.cpu().numpy()

        for i in range(len(label_ids)):
            pred_labels = []
            true_label = []
            for j in range(len(label_ids[i])):
                if label_ids[i][j] != -100:
                    true_label.append(label_ids[i][j])
                    pred_label = np.argmax(logits[i][j])
                    pred_labels.append(pred_label)
            predictions.append(pred_labels)
            true_labels.append(true_label)


Epoch 1/8, Loss: 0.2713
Epoch 2/8, Loss: 0.1151
Epoch 3/8, Loss: 0.0883
Epoch 4/8, Loss: 0.0717
Epoch 5/8, Loss: 0.0592
Epoch 6/8, Loss: 0.0508
Epoch 7/8, Loss: 0.0428
Epoch 8/8, Loss: 0.0392


In [ ]:
# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

In [ ]:
# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

DISTILROBERTA

Precision: 0.9730
Recall: 0.9162
F1-Score: 0.9325


In [13]:
!pip install seqeval scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
